# 2.1 Create Face Dataset Splits

Purpose:
- Load the cleaned face manifest
- Create stratified train/validation/test splits
- Save split manifest
- Verify class balance per split

In [1]:
import pandas as pd

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FACE_CLEAN_MANIFEST_PATH,
    FACE_SPLIT_MANIFEST_PATH,
)
from rural_stroke_assist.preprocessing.splitting import (
    add_stratified_split,
    summarize_split_balance,
)

In [2]:
face_clean_df = pd.read_csv(FACE_CLEAN_MANIFEST_PATH)

face_clean_df.head()

,path,file_name,class_label,extension,readable,width,height,file_hash,is_clean
0,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0022.jpg,NonStroke,.jpg,True,299,169,a3e6d954dd74d17e89c497d4967c93366d623b4b5b694f...,True
1,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0024.jpg,NonStroke,.jpg,True,379,379,b2a1a613cdb309d96a998ed46adea5fe182aa661a40c88...,True
2,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0025.jpg,NonStroke,.jpg,True,275,183,7038e96c4cafff98b47b9df614b9cd51473476e4d99699...,True
3,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0027.jpg,NonStroke,.jpg,True,163,222,b9a1bd4960d3afa939c7f908c1d821c468ebd247e92f50...,True
4,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0030.jpg,NonStroke,.jpg,True,342,382,bac0af9d5eef5722d3b989d1e29677f6d4b479bda29748...,True


In [3]:
face_split_df = add_stratified_split(
    df=face_clean_df,
    label_column="class_label",
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=42,
)

face_split_df.head()

,path,file_name,class_label,extension,readable,width,height,file_hash,is_clean,split
0,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0022.jpg,NonStroke,.jpg,True,299,169,a3e6d954dd74d17e89c497d4967c93366d623b4b5b694f...,True,train
1,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0024.jpg,NonStroke,.jpg,True,379,379,b2a1a613cdb309d96a998ed46adea5fe182aa661a40c88...,True,test
2,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0025.jpg,NonStroke,.jpg,True,275,183,7038e96c4cafff98b47b9df614b9cd51473476e4d99699...,True,train
3,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0027.jpg,NonStroke,.jpg,True,163,222,b9a1bd4960d3afa939c7f908c1d821c468ebd247e92f50...,True,train
4,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0030.jpg,NonStroke,.jpg,True,342,382,bac0af9d5eef5722d3b989d1e29677f6d4b479bda29748...,True,test


In [4]:
split_summary = summarize_split_balance(
    df=face_split_df,
    split_column="split",
    label_column="class_label",
)

split_summary

,split,class_label,count,percentage
0,test,NonStroke,208,65.41
1,test,Stroke,110,34.59
2,train,NonStroke,970,65.54
3,train,Stroke,510,34.46
4,val,NonStroke,208,65.62
5,val,Stroke,109,34.38


In [5]:
face_split_df["split"].value_counts()

split
train    1480
test      318
val       317
Name: count, dtype: int64

In [6]:
FACE_SPLIT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

face_split_df.to_csv(FACE_SPLIT_MANIFEST_PATH, index=False)

print("Saved face split manifest to:", FACE_SPLIT_MANIFEST_PATH)

Saved face split manifest to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\data\processed\face_split_manifest.csv


## Split Decision

The cleaned face dataset was split using stratified sampling to preserve the Stroke/NonStroke class ratio across train, validation, and test sets.

Because exact duplicate images were removed before splitting, this reduces the risk of duplicate-based data leakage.

Remaining limitation:
The dataset may still contain visually similar images or same-person images that cannot be fully detected from the current metadata.